# ChatBot EcoMarket — v2.2.0
**Evolución: PostgreSQL + Groq + RAG + flujo comercial MVP**

| Campo | Valor |
|---|---|
| Versión | 2.2.0 |
| Base anterior | ChatBot_2.1.0.ipynb |
| Motor de datos | PostgreSQL 18 / NEON |
| Arquitectura | `RetailService` como única capa de acceso a SQL |
| Modelo NLP | `mDeBERTa-v3` zero-shot + reglas de intención |
| LLM auxiliar | Groq API |
| RAG | Documental, orientado a políticas y soporte |
| Enfoque actual | MVP comercial conversacional |

### Changelog vs 2.1.0
- **Checkout mejorado**: ahora el flujo pide datos de cliente y luego método de pago antes de confirmar el pedido.
- **Métodos de pago MVP**: se simplifica a `tarjeta`, `efectivo` y `contra entrega`.
- **Mensajería operativa de pago**:
  - `tarjeta` y `efectivo`: el bot indica acercarse a tienda para completar el pago.
  - `contra entrega`: el bot informa que el cobro se realiza al entregar el pedido.
- **Promociones automáticas simples**:
  - 5 % de descuento por volumen desde 3 unidades del mismo producto.
  - envío gratis desde 50 €.
  - 10 % de descuento adicional sobre el carrito desde 75 €.
- **Resumen comercial del carrito**: el bot ya calcula beneficios aplicados y total estimado del pedido.
- **Sustitución comercial**:
  - si un producto está agotado, recomienda alternativas similares;
  - si el producto no existe exactamente en catálogo, propone productos parecidos disponibles.
- **RAG documental refinado**: se actualizan documentos de pagos, promociones y FAQ para responder con contexto controlado.
- **Respuesta final de compra mejorada**: incluye número de pedido, método de pago, items, beneficios aplicados e importe total.
- **Persistencia**: se mantiene PostgreSQL / NEON como fuente operativa de verdad.
- **Seguridad**: se mantiene `RetailService` como única capa de acceso parametrizado a SQL.

### Capacidades actuales
- Consulta de disponibilidad por producto o categoría.
- Compra conversacional de varios productos en una sola sesión.
- Carrito de compras multiartículo.
- Validación de stock y contrapropuesta cuando no hay suficiente inventario.
- Captura de nombre, email y teléfono para generar pedidos.
- Selección de método de pago dentro del flujo conversacional.
- Creación de pedido único con múltiples líneas.
- Consulta de estado de pedido e incidencias.
- RAG documental para políticas, promociones, pagos y preguntas frecuentes.
- Logging del chatbot en PostgreSQL.

### Secciones
1. Configuración  
2. Conexión PostgreSQL  
3. RetailService  
4. Utilidades NLP  
5. RAG documental  
6. Reglas de intención  
7. Generación de respuestas y carrito  
8. Logging del chatbot  
9. Tests conversacionales  
10. Chat interactivo

## 1. Configuración

In [30]:
!pip install psycopg2-binary python-dotenv transformers torch



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# ============================================================
# 1. CONFIGURACION GENERAL
# ============================================================
# Dependencias: pip install psycopg2-binary python-dotenv transformers torch
# Credenciales: crea un archivo .env en la misma carpeta que este notebook:
#
#
#
# NUNCA subas el .env al repositorio. Añádelo a .gitignore.
# ============================================================

import os
import re
import json
import urllib.request
import urllib.error
from pathlib import Path
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, List, Dict, Any

# python-dotenv carga .env si existe; si no, usa las variables de entorno del sistema
def _cargar_env_manual(ruta_env: Path) -> None:
    """Carga un .env simple sin depender de python-dotenv."""
    for line in ruta_env.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


env_path = None
CANDIDATOS_ENV = [
    Path(r"C:\data_sciences\GIT\ChatBot_EcoMarket\ChatBot_EcoMarket\Bot\.env"),
    Path.cwd() / ".env",
    Path.cwd() / "Bot" / ".env",
    Path.cwd() / "ChatBot_EcoMarket" / "Bot" / ".env",
]
for base in [Path.cwd(), *Path.cwd().parents]:
    CANDIDATOS_ENV.extend([
        base / ".env",
        base / "Bot" / ".env",
        base / "ChatBot_EcoMarket" / "Bot" / ".env",
    ])

vistos = set()
for candidate in CANDIDATOS_ENV:
    candidate = candidate.resolve()
    if str(candidate) in vistos:
        continue
    vistos.add(str(candidate))
    if candidate.exists():
        env_path = candidate
        break

if env_path:
    try:
        from dotenv import load_dotenv
        load_dotenv(env_path, override=True)
        print(f"✓ .env cargado con python-dotenv desde {env_path}")
    except ImportError:
        _cargar_env_manual(env_path)
        print(f"✓ .env cargado manualmente desde {env_path}")
else:
    print("⚠ No se encontró .env. Usando variables de entorno del sistema.")

# ── Conexión PostgreSQL ──────────────────────────────────────────────────────
DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = int(os.getenv("DB_PORT", "5432"))
DB_NAME     = os.getenv("DB_NAME",     "ChatBot_Ecomarket")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_SSLMODE = os.getenv("DB_SSLMODE", "require")

if not DB_PASSWORD.strip():
    print("⚠ DB_PASSWORD está vacío. Revisa/crea el archivo .env junto al notebook.")

# ── NLP ─────────────────────────────────────────────────────────────────────
MODEL_NAME           = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
MODEL_LOCAL_ONLY     = False
CONFIDENCE_THRESHOLD = 0.55
HIGH_CONFIDENCE_THRESHOLD = 0.82
INTENCION_SOPORTE    = "Hablar con soporte humano"

# ── Groq ───────────────────────────────────────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_API_BASE = os.getenv("GROQ_API_BASE", "https://api.groq.com/openai/v1/chat/completions")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_ENABLED = bool(GROQ_API_KEY.strip())
GROQ_DISABLED_REASON = ""
GROQ_LAST_ERROR = ""
GROQ_TIMEOUT_SECONDS = int(os.getenv("GROQ_TIMEOUT_SECONDS", "20"))
GROQ_PREFER_BELOW_SCORE = float(os.getenv("GROQ_PREFER_BELOW_SCORE", "0.72"))
GROQ_TEMPERATURE_INTENT = float(os.getenv("GROQ_TEMPERATURE_INTENT", "0.10"))
GROQ_TEMPERATURE_EXTRACTION = float(os.getenv("GROQ_TEMPERATURE_EXTRACTION", "0.05"))
GROQ_TEMPERATURE_RAG = float(os.getenv("GROQ_TEMPERATURE_RAG", "0.15"))
GROQ_ENV_PATH_CARGADO = str(env_path) if "env_path" in locals() and env_path else ""

ETIQUETAS_NEGOCIO = [
    "Estado del pedido",
    "Incidencia con pedido incompleto o no recibido",
    "Devoluciones y cambios",
    "Producto dañado caducado o en mal estado",
    "Disponibilidad de productos",
    "Compra de productos",
    "Promociones y cupones",
    "Métodos de pago y compra",
    INTENCION_SOPORTE,
]

# ── Negocio ──────────────────────────────────────────────────────────────────
VERSION               = "2.2.0"
CANTIDAD_POR_DEFECTO  = 1
COMANDOS_SALIDA       = {"salir", "escape", "exit", "quit", "cerrar"}
METODOS_PAGO_VALIDOS  = {"tarjeta", "efectivo", "contra_entrega"}
UMBRAL_ENVIO_GRATIS   = 50.0
UMBRAL_DESCUENTO_CARRITO = 75.0
PORCENTAJE_DESCUENTO_CARRITO = 0.10
UMBRAL_DESCUENTO_VOLUMEN = 3
PORCENTAJE_DESCUENTO_VOLUMEN = 0.05

POLITICAS = {
    "devoluciones": (
        "Las devoluciones pueden solicitarse hasta 30 días después de la compra, "
        "presentando ticket o número de pedido. En productos frescos, perecederos "
        "o de higiene pueden aplicar restricciones."
    ),
    "promociones": (
        "EcoMarket aplica promociones automáticas sencillas para el MVP. "
        "A partir de 3 unidades del mismo producto puede aplicarse un 5% de descuento por volumen. "
        "Si el carrito supera 50 euros, el pedido obtiene envío gratis. "
        "Si el carrito alcanza 75 euros o más, se aplica un 10% de descuento adicional sobre el total promocional."
    ),
    "pagos": (
        "Los métodos de pago del MVP son tarjeta, efectivo y contra entrega. "
        "Si eliges efectivo o tarjeta, el chatbot te indicará que debes acercarte a la tienda para completar el pago. "
        "Si eliges contra entrega, el cobro se realiza al entregar el pedido."
    ),
}

# ── Tablas mínimas requeridas ─────────────────────────────────────────────────
TABLAS_REQUERIDAS = [
    "categorias", "productos", "inventario",
    "clientes", "pedidos", "detalle_pedidos",
    "movimientos_inventario", "logs_chatbot",
]

print(f"ChatBot EcoMarket v{VERSION} — configuración cargada")
print(f"DB target: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print(f"Groq: {'habilitado' if GROQ_ENABLED else 'sin GROQ_API_KEY'} | modelo={GROQ_MODEL}")
print(f"Groq env path: {GROQ_ENV_PATH_CARGADO or 'no detectado'}")


✓ .env cargado desde c:\data_sciences\GIT\ChatBot_EcoMarket\ChatBot_EcoMarket\Bot\.env
ChatBot EcoMarket v2.2.0 — configuración cargada
DB target: neondb_owner@ep-restless-cell-al2diw6p-pooler.c-3.eu-central-1.aws.neon.tech:5432/neondb
Groq: habilitado | modelo=llama-3.3-70b-versatile


## 2. Conexión PostgreSQL

In [32]:
# ============================================================
# 2. CONEXION POSTGRESQL
# ============================================================

import psycopg2
from psycopg2.extras import RealDictCursor

DB_SSLMODE = os.getenv("DB_SSLMODE", "require")

def get_connection():
    """Devuelve una conexión nueva a PostgreSQL. Lanza excepción si falla."""
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        sslmode=DB_SSLMODE,
        connect_timeout=20,
    )

def probar_conexion():
    """Verifica que la conexión a la base de datos es posible."""
    try:
        with get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT version();")
                version = cur.fetchone()[0]
        print(f"✓ Conexión OK → {version[:60]}")
        return True
    except psycopg2.OperationalError as exc:
        print(f"✗ Error de conexión: {exc}")
        print("  Revisa DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD en tu .env")
        return False

probar_conexion()


✓ Conexión OK → PostgreSQL 18.4 (365f1e4) on aarch64-unknown-linux-gnu, comp


True

In [33]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user, now();")
        print(cur.fetchone())

('neondb', 'neondb_owner', datetime.datetime(2026, 6, 3, 18, 14, 26, 765823, tzinfo=datetime.timezone.utc))


In [34]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        for row in cur.fetchall():
            print(row[0])

categorias
clientes
detalle_pedidos
inventario
logs_chatbot
movimientos_inventario
pedidos
productos
v_alertas_stock
v_detalle_pedido_completo
v_pedidos_resumen
v_stock_actual


## 3. RetailService
Capa única de acceso a PostgreSQL. El chatbot nunca escribe SQL directamente.

In [35]:
# ============================================================
# 3. RETAIL SERVICE
# Toda consulta o escritura a PostgreSQL pasa por esta clase.
# El notebook solo llama a métodos de RetailService.
# ============================================================

class RetailService:
    """
    Capa de acceso a datos para EcoMarket sobre PostgreSQL.
    Usa consultas parametrizadas en todos los métodos para evitar SQL injection.
    """

    # ── Cache en memoria (se recarga con refresh_cache) ──────────────────────
    _categorias: List[str] = []
    _catalogo:   Dict[str, Dict] = {}   # {codigo_producto: {nombre, aliases, stock, ...}}

    # ── Conexión ─────────────────────────────────────────────────────────────
    def validar_conexion(self) -> bool:
        """Comprueba que la BD es accesible. Devuelve True/False."""
        return probar_conexion()

    def _query(self, sql: str, params=None) -> List[Dict]:
        """Ejecuta una SELECT y devuelve lista de dicts. Uso interno."""
        with get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql, params)
                return [dict(row) for row in cur.fetchall()]

    def _execute(self, sql: str, params=None) -> None:
        """Ejecuta INSERT/UPDATE sin devolver filas. Uso interno."""
        with get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, params)
            conn.commit()

    # ── Validación de esquema ─────────────────────────────────────────────────
    def obtener_esquema_tablas(self) -> Dict[str, List[str]]:
        """
        Consulta information_schema para obtener columnas reales de cada tabla.
        Útil para detectar diferencias entre lo esperado y lo desplegado.
        """
        sql = """
            SELECT table_name, column_name
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = ANY(%s)
            ORDER BY table_name, ordinal_position
        """
        rows = self._query(sql, (TABLAS_REQUERIDAS,))
        esquema: Dict[str, List[str]] = {}
        for row in rows:
            esquema.setdefault(row["table_name"], []).append(row["column_name"])
        return esquema

    def validar_contrato_minimo(self) -> bool:
        """
        Verifica que todas las tablas requeridas existen en la BD.
        Imprime un reporte y devuelve True si todo está OK.
        """
        esquema = self.obtener_esquema_tablas()
        ok = True
        print("── Validación de esquema ────────────────────────")
        for tabla in TABLAS_REQUERIDAS:
            if tabla in esquema:
                print(f"  ✓ {tabla}  ({len(esquema[tabla])} columnas)")
            else:
                print(f"  ✗ {tabla}  — TABLA NO ENCONTRADA")
                ok = False
        print("─────────────────────────────────────────────────")
        if ok:
            print("✓ Contrato mínimo cumplido.")
        else:
            print("✗ Hay tablas faltantes. Revisa el script SQL de creación.")
        return ok

    # ── Caché de catálogo ─────────────────────────────────────────────────────
    def refresh_cache(self) -> None:
        """
        Recarga categorías y catálogo de productos desde PostgreSQL.
        Llámalo al inicio y después de cualquier cambio en productos/inventario.
        """
        # Categorías
        rows = self._query(
            "SELECT nombre FROM categorias WHERE activo = TRUE ORDER BY nombre"
        )
        self._categorias = [r["nombre"] for r in rows]

        # Catálogo completo con stock
        rows = self._query("""
            SELECT
                p.codigo_producto,
                p.nombre,
                p.precio,
                p.aliases,
                p.activo,
                c.nombre AS categoria,
                i.stock_actual,
                i.stock_minimo
            FROM productos p
            JOIN categorias c ON c.categoria_id = p.categoria_id
            JOIN inventario i ON i.producto_id  = p.producto_id
            WHERE p.activo = TRUE
        """)

        self._catalogo = {}
        for r in rows:
            aliases_raw = r["aliases"] or ""
            aliases = [a.strip() for a in aliases_raw.split("|") if a.strip()]
            nombre = r["nombre"].strip()
            if nombre not in aliases:
                aliases.insert(0, nombre)
            self._catalogo[r["codigo_producto"]] = {
                "nombre":      nombre,
                "categoria":   r["categoria"],
                "precio":      float(r["precio"]),
                "stock":       int(r["stock_actual"]),
                "stock_minimo": int(r["stock_minimo"]),
                "aliases":     aliases,
            }

        print(f"✓ Caché recargada: {len(self._categorias)} categorías, "
              f"{len(self._catalogo)} productos")

    # ── Categorías ────────────────────────────────────────────────────────────
    def listar_categorias(self) -> List[str]:
        """Devuelve lista de categorías activas."""
        return list(self._categorias)

    def buscar_categoria(self, texto: str) -> Optional[str]:
        """
        Busca si el texto del usuario menciona alguna categoría existente.
        Devuelve el nombre de la categoría o None.
        """
        texto_n = _normalizar(texto)
        for cat in self._categorias:
            if _normalizar(cat) in texto_n:
                return cat
        return None

    # ── Productos ─────────────────────────────────────────────────────────────
    def buscar_producto(self, texto: str) -> Optional[str]:
        """
        Busca un producto en el catálogo usando aliases.
        Devuelve el codigo_producto o None.
        Prioridad: alias exacto > alias parcial.
        """
        texto_n = _normalizar(texto)
        # Evitar falsos positivos de categoría
        if self.buscar_categoria(texto):
            return None
        for codigo, datos in self._catalogo.items():
            for alias in datos["aliases"]:
                if _normalizar(alias) in texto_n:
                    return codigo
        return None

    def consultar_productos_por_categoria(self, categoria: str) -> List[Dict]:
        """
        Devuelve productos disponibles (stock > 0) de una categoría.
        Usa la función SQL fn_productos_categoria.
        """
        rows = self._query(
            "SELECT * FROM fn_productos_categoria(%s)",
            (categoria,)
        )
        return rows

    def consultar_stock_producto(self, codigo_producto: str) -> Optional[Dict]:
        """Devuelve el stock actual de un producto por su código."""
        rows = self._query(
            "SELECT * FROM v_stock_actual WHERE codigo_producto = %s",
            (codigo_producto,)
        )
        return rows[0] if rows else None

    # ── Clientes ─────────────────────────────────────────────────────────────
    def consultar_cliente(self, codigo_cliente: str) -> Optional[Dict]:
        """Devuelve un cliente por su código visible."""
        rows = self._query(
            "SELECT * FROM clientes WHERE codigo_cliente = %s",
            (codigo_cliente.upper(),)
        )
        return rows[0] if rows else None

    def _normalizar_nombre_cliente(self, nombre: str) -> str:
        """Normaliza nombre para comparar clientes sin mayúsculas ni espacios extra."""
        return re.sub(r"\s+", " ", nombre.strip().lower())

    def buscar_cliente_por_nombre(self, nombre: str) -> Optional[Dict]:
        """Busca un cliente por nombre normalizado para mitigar duplicados."""
        nombre_n = self._normalizar_nombre_cliente(nombre)
        rows = self._query("""
            SELECT *
            FROM clientes
            WHERE LOWER(REGEXP_REPLACE(TRIM(COALESCE(nombre, '')), '\\s+', ' ', 'g')) = %s
            ORDER BY codigo_cliente
            LIMIT 1
        """, (nombre_n,))
        return rows[0] if rows else None

    def obtener_o_crear_cliente_por_contacto(self, nombre: str, email: Optional[str] = None, telefono: Optional[str] = None) -> Dict:
        """Devuelve cliente existente o crea uno, guardando email/teléfono si vienen informados."""
        nombre_limpio = re.sub(r"\s+", " ", (nombre or "").strip())
        email_limpio = (email or "").strip().lower() or None
        telefono_limpio = re.sub(r"\s+", "", (telefono or "").strip()) or None

        existente = None
        if email_limpio:
            rows = self._query("SELECT * FROM clientes WHERE LOWER(COALESCE(email, '')) = %s LIMIT 1", (email_limpio,))
            existente = rows[0] if rows else None
        if not existente and telefono_limpio:
            rows = self._query("SELECT * FROM clientes WHERE REGEXP_REPLACE(COALESCE(telefono, ''), '\\s+', '', 'g') = %s LIMIT 1", (telefono_limpio,))
            existente = rows[0] if rows else None
        if not existente and nombre_limpio:
            existente = self.buscar_cliente_por_nombre(nombre_limpio)

        if existente:
            self._execute("""
                UPDATE clientes
                   SET nombre = COALESCE(NULLIF(%s, ''), nombre),
                       email = COALESCE(%s, email),
                       telefono = COALESCE(%s, telefono)
                 WHERE cliente_id = %s
            """, (nombre_limpio, email_limpio, telefono_limpio, existente["cliente_id"]))
            actualizado = self.consultar_cliente(existente["codigo_cliente"])
            actualizado["creado"] = False
            return actualizado

        rows = self._query("""
            SELECT COALESCE(MAX((REGEXP_MATCH(codigo_cliente, '^CLI-(\\d+)$'))[1]::INT), 0) + 1 AS siguiente
            FROM clientes
        """)
        codigo_cliente = f"CLI-{rows[0]['siguiente']:03d}"

        rows = self._query("""
            INSERT INTO clientes (codigo_cliente, nombre, email, telefono)
            VALUES (%s, %s, %s, %s)
            RETURNING *
        """, (codigo_cliente, nombre_limpio, email_limpio, telefono_limpio))
        cliente = rows[0]
        cliente["creado"] = True
        return cliente

    def obtener_o_crear_cliente_por_nombre(self, nombre: str) -> Dict:
        """Compatibilidad: crea/busca cliente solo con nombre."""
        return self.obtener_o_crear_cliente_por_contacto(nombre=nombre)

    # ── Pedidos ───────────────────────────────────────────────────────────────
    def generar_codigo_pedido(self) -> str:
        """Genera un codigo PED-XXX desde la BD, evitando colisiones con pedidos existentes."""
        rows = self._query("""
            SELECT COALESCE(MAX((REGEXP_MATCH(codigo_pedido, '^PED-(\\d+)$'))[1]::INT), 0) + 1 AS siguiente
            FROM pedidos
            WHERE codigo_pedido ~ '^PED-\\d+$'
        """)
        return f"PED-{int(rows[0]['siguiente']):03d}"

    def consultar_pedido(self, codigo_pedido: str) -> Optional[Dict]:
        """Devuelve la cabecera de un pedido por su código visible."""
        rows = self._query(
            "SELECT * FROM v_pedidos_resumen WHERE codigo_pedido = %s",
            (codigo_pedido.upper(),)
        )
        return rows[0] if rows else None

    def consultar_detalle_pedido(self, codigo_pedido: str) -> List[Dict]:
        """Devuelve las líneas del pedido con nombre de producto y subtotales."""
        return self._query(
            "SELECT * FROM v_detalle_pedido_completo WHERE codigo_pedido = %s",
            (codigo_pedido.upper(),)
        )

    def consultar_movimientos_pedido(self, codigo_pedido: str) -> List[Dict]:
        """Devuelve los movimientos de inventario asociados a un pedido."""
        return self._query("""
            SELECT
                m.fecha_hora,
                m.tipo_movimiento,
                p.nombre AS producto,
                m.cantidad,
                m.stock_anterior,
                m.stock_nuevo,
                m.motivo
            FROM movimientos_inventario m
            JOIN pedidos  pe ON pe.pedido_id  = m.pedido_id
            JOIN productos p ON p.producto_id = m.producto_id
            WHERE pe.codigo_pedido = %s
            ORDER BY m.fecha_hora
        """, (codigo_pedido.upper(),))

    # ── Operaciones de escritura ───────────────────────────────────────────────
    def crear_pedido(
        self,
        codigo_cliente: str,
        carrito: List[Dict],   # [{"codigo_producto": "PROD-001", "cantidad": 2}]
        canal: str = "chatbot",
        metodo_pago: Optional[str] = None,
        observaciones: Optional[str] = None,
    ) -> Dict:
        """
        Crea un pedido completo:
          1. Valida stock de cada línea.
          2. Genera codigo_pedido secuencial en PostgreSQL.
          3. Llama a fn_crear_pedido y fn_registrar_compra por cada línea.
        Devuelve {"ok": bool, "codigo_pedido": str, "errores": [...]}
        """
        errores = []
        # 1. Validar stock de todo el carrito antes de crear nada
        for item in carrito:
            rows = self._query(
                "SELECT * FROM fn_validar_stock(%s, %s)",
                (item["codigo_producto"], item["cantidad"])
            )
            if rows and not rows[0]["disponible"]:
                errores.append(rows[0]["mensaje"])

        if errores:
            return {"ok": False, "codigo_pedido": None, "errores": errores}

        # 2. Generar código de pedido con reintento por si existe una colisión
        codigo_pedido = None
        rows = []
        for _ in range(5):
            codigo_pedido = self.generar_codigo_pedido()
            rows = self._query(
                "SELECT * FROM fn_crear_pedido(%s, %s, %s, %s, %s)",
                (codigo_pedido, codigo_cliente, canal, metodo_pago, observaciones)
            )
            if rows and rows[0]["exito"]:
                break
            if rows and "existe" not in str(rows[0].get("mensaje", "")).lower():
                return {"ok": False, "codigo_pedido": None,
                        "errores": [rows[0]["mensaje"]]}
        else:
            return {"ok": False, "codigo_pedido": None,
                    "errores": ["No fue posible generar un código de pedido único."]}

        # 3. Crear cabecera OK, continuar con líneas

        # 4. Registrar compra por cada línea
        for item in carrito:
            self._query(
                "SELECT * FROM fn_registrar_compra(%s, %s, %s)",
                (codigo_pedido, item["codigo_producto"], item["cantidad"])
            )

            # Añadir línea en detalle_pedidos
            self._execute("""
                INSERT INTO detalle_pedidos
                    (pedido_id, producto_id, cantidad_comprada, precio_unitario, estado_linea)
                SELECT pe.pedido_id, pr.producto_id, %s, pr.precio, 'pendiente'
                FROM pedidos pe, productos pr
                WHERE pe.codigo_pedido = %s AND pr.codigo_producto = %s
            """, (item["cantidad"], codigo_pedido, item["codigo_producto"]))

        # Refrescar caché de stock
        self.refresh_cache()
        return {"ok": True, "codigo_pedido": codigo_pedido, "errores": []}

    def registrar_devolucion(
        self,
        codigo_pedido: str,
        codigo_producto: str,
        cantidad: int,
        motivo: str = "devolucion cliente",
    ) -> Dict:
        """Registra una devolución y suma stock. Devuelve resultado de la función SQL."""
        rows = self._query(
            "SELECT * FROM fn_registrar_devolucion(%s, %s, %s, %s)",
            (codigo_pedido.upper(), codigo_producto, cantidad, motivo)
        )
        self.refresh_cache()
        return rows[0] if rows else {"exito": False, "mensaje": "Error desconocido"}

    # ── Logging ───────────────────────────────────────────────────────────────
    def guardar_log_chatbot(
        self,
        pregunta_cliente: str,
        intencion_detectada: str,
        confianza: float,
        origen_intencion: str,
        respuesta_bot: str,
        intencion_correcta: str = "",
        respuesta_correcta: str = "",
        aprobado_para_entrenamiento: bool = False,
        notas: str = "",
    ) -> None:
        """Inserta un registro en logs_chatbot en PostgreSQL."""
        self._execute("""
            INSERT INTO logs_chatbot (
                pregunta_cliente, intencion_detectada, intencion_correcta,
                confianza, origen_intencion, respuesta_bot, respuesta_correcta,
                aprobado_para_entrenamiento, notas
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            pregunta_cliente, intencion_detectada, intencion_correcta,
            round(confianza, 4), origen_intencion, respuesta_bot, respuesta_correcta,
            aprobado_para_entrenamiento, notas,
        ))


### Inicializar RetailService

In [ ]:
# ── Inicialización ───────────────────────────────────────────────────────────
# Si la conexión falla, el chatbot seguirá funcionando en modo degradado
# usando reglas de intención y respondiendo sin datos de inventario.

service = RetailService()
if service.validar_conexion():
    service.validar_contrato_minimo()
    service.refresh_cache()
else:
    print("⚠ Chatbot en modo degradado — sin conexión a PostgreSQL")
    print("  Revisa las credenciales en tu archivo .env")


✓ Conexión OK → PostgreSQL 18.4 (365f1e4) on aarch64-unknown-linux-gnu, comp


## 4. Utilidades NLP

In [ ]:
# ============================================================
# 4. UTILIDADES NLP
# Funciones de texto que usa el motor de intenciones.
# No contienen SQL; leen del caché de RetailService.
# ============================================================

def _normalizar(texto: str) -> str:
    """Pasa a minúsculas, elimina acentos y espacios extra."""
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def extraer_id_pedido(mensaje: str) -> Optional[str]:
    """Extrae un código tipo PED-123 del mensaje del usuario."""
    match = re.search(r"\bped[-\s]?\d+\b", mensaje, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group().replace(" ", "-").upper()


def extraer_codigo_cliente(mensaje: str) -> Optional[str]:
    """Extrae un código tipo CLI-001 del mensaje del usuario."""
    match = re.search(r"\bcli[-\s]?\d+\b", mensaje, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group().replace(" ", "-").upper()


def extraer_cantidad_solicitada(mensaje: str) -> int:
    """Extrae la cantidad numérica solicitada o devuelve CANTIDAD_POR_DEFECTO."""
    msg_n = _normalizar(mensaje)
    patrones = [
        r"\b(?:quiero|necesito|comprar|llevar|pedir|solicito)\s+(\d+)\b",
        r"\b(\d+)\s+(?:unidades|uds|piezas|kilos|kg|paquetes)?\b",
    ]
    for patron in patrones:
        m = re.search(patron, msg_n)
        if m:
            cantidad = int(m.group(1))
            if cantidad > 0:
                return cantidad
    return CANTIDAD_POR_DEFECTO


def tiene_cantidad_explicita(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return bool(re.search(r"\b\d+\b", msg_n))


def contiene_alguna(texto_n: str, expresiones: List[str]) -> bool:
    return any(e in texto_n for e in expresiones)


def es_respuesta_afirmativa(mensaje: str) -> bool:
    return _normalizar(mensaje) in {"si", "sí", "claro", "vale", "ok", "dale", "por favor", "exacto", "confirmo", "confirmar"}


def es_respuesta_negativa(mensaje: str) -> bool:
    return _normalizar(mensaje) in {"no", "nop", "cancelar", "cancela", "mejor no", "anular", "vaciar"}


def es_solicitud_finalizar_compra(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["finalizar", "terminar compra", "cerrar compra", "checkout", "pagar", "confirmar compra", "generar pedido"])


def es_solicitud_ver_carrito(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["ver carrito", "mi carrito", "carrito", "resumen de compra"])


def es_solicitud_vaciar_carrito(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["vaciar carrito", "limpiar carrito", "cancelar carrito", "borrar carrito"])


def es_solicitud_compra(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["comprar", "llevar", "pedir", "solicito", "agregar", "añadir", "anadir", "sumar", "quiero", "necesito", "venderme"])


def es_consulta_exploratoria(mensaje: str) -> bool:
    """
    Detecta mensajes abiertos donde conviene dar más espacio a Groq
    y menos a las reglas o al estado pendiente del carrito.
    """
    msg_n = _normalizar(mensaje)
    pistas = [
        "no se bien por donde empezar",
        "no sé bien por dónde empezar",
        "organizando la compra",
        "compra de la semana",
        "ayudame a armar",
        "ayúdame a armar",
        "que me recomiendas",
        "qué me recomiendas",
        "no tengo claro",
        "lo de siempre",
        "para la casa",
        "varias cosas del mercado",
        "algo para",
    ]
    if not contiene_alguna(msg_n, pistas):
        return False
    if extraer_id_pedido(mensaje):
        return False
    if tiene_cantidad_explicita(mensaje):
        return False
    if service.buscar_producto(mensaje) or service.buscar_categoria(mensaje):
        return False
    return True


def parece_nombre_completo(mensaje: str) -> bool:
    partes = re.sub(r"\s+", " ", mensaje.strip()).split(" ")
    return len([p for p in partes if len(p) >= 2]) >= 2


def necesita_soporte_por_palabras_clave(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, [
        "humano", "agente", "persona", "reclamo", "queja", "urgente",
        "denuncia", "no me ayudan"
    ])


def quiere_ver_categorias(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["categoria", "categorias", "otras categorias", "ver categorias"])


def extraer_contacto_local(mensaje: str) -> Dict[str, Optional[str]]:
    """Extrae contacto con regex simples; Groq puede completar si el texto es más natural."""
    email_match = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", mensaje)
    phone_match = re.search(r"(?:\+?\d[\d\s().-]{7,}\d)", mensaje)
    nombre = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " ", mensaje)
    nombre = re.sub(r"(?:\+?\d[\d\s().-]{7,}\d)", " ", nombre)
    nombre = re.sub(r"\b(?:mi nombre es|soy|me llamo|telefono|teléfono|email|correo|móvil|movil|celular|es|mi)\b", " ", nombre, flags=re.IGNORECASE)
    nombre = re.sub(r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ\s'-]", " ", nombre)
    nombre = re.sub(r"\s+", " ", nombre).strip()
    if len(nombre.split()) < 2:
        nombre = ""
    return {
        "nombre": nombre or None,
        "email": email_match.group(0).lower() if email_match else None,
        "telefono": re.sub(r"\s+", "", phone_match.group(0)) if phone_match else None,
    }


def validar_extraccion_operativa(data: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Normaliza la salida de Groq a un contrato operativo pequeño y seguro."""
    data = data or {}
    cliente = data.get("cliente") if isinstance(data.get("cliente"), dict) else {}
    items = data.get("items") if isinstance(data.get("items"), list) else []
    limpio = {
        "intencion": str(data.get("intencion") or "").strip(),
        "accion": str(data.get("accion") or "").strip(),
        "cliente": {
            "nombre": cliente.get("nombre") or None,
            "email": cliente.get("email") or None,
            "telefono": cliente.get("telefono") or None,
        },
        "items": [],
    }
    for item in items:
        if not isinstance(item, dict):
            continue
        producto = str(item.get("producto") or "").strip()
        try:
            cantidad = int(item.get("cantidad") or CANTIDAD_POR_DEFECTO)
        except Exception:
            cantidad = CANTIDAD_POR_DEFECTO
        if producto and cantidad > 0:
            limpio["items"].append({"producto": producto, "cantidad": cantidad})
    return limpio


def _llamar_groq_chat(messages: List[Dict[str, str]], max_completion_tokens: int, temperature: float = 0.0, feature_name: str = "Groq") -> Optional[str]:
    """Centraliza llamadas a Groq y deja trazabilidad útil sin ensuciar el chat."""
    global GROQ_ENABLED, GROQ_DISABLED_REASON, GROQ_LAST_ERROR
    if not GROQ_ENABLED:
        return None

    payload = {
        "model": GROQ_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_completion_tokens": max_completion_tokens,
    }
    req = urllib.request.Request(
        GROQ_API_BASE,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json", "User-Agent": "groq-python/0.9.0"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=GROQ_TIMEOUT_SECONDS) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        GROQ_DISABLED_REASON = ""
        GROQ_LAST_ERROR = ""
        return data["choices"][0]["message"]["content"].strip()
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", "replace")
        GROQ_LAST_ERROR = f"HTTP {exc.code}: {body[:400]}"
        if exc.code in (401, 403):
            GROQ_ENABLED = False
            GROQ_DISABLED_REASON = f"Groq deshabilitado en esta sesión por error de autenticación/autorización ({exc.code})."
            print(f"  ⚠ {GROQ_DISABLED_REASON}")
        else:
            print(f"  ⚠ {feature_name} no disponible temporalmente: HTTP {exc.code}.")
        return None
    except Exception as exc:
        GROQ_LAST_ERROR = str(exc)
        print(f"  ⚠ {feature_name} no disponible temporalmente: {exc}")
        return None


def probar_conexion_groq() -> Dict[str, Any]:
    """Chequeo rápido de Groq desde el propio notebook."""
    contenido = _llamar_groq_chat(
        [
            {"role": "system", "content": "Responde solo ok."},
            {"role": "user", "content": "test"},
        ],
        max_completion_tokens=20,
        temperature=0.0,
        feature_name="Prueba de conexión Groq",
    )
    return {
        "groq_enabled": GROQ_ENABLED,
        "modelo": GROQ_MODEL,
        "respuesta": contenido,
        "disabled_reason": GROQ_DISABLED_REASON,
        "last_error": GROQ_LAST_ERROR,
    }


def diagnostico_groq() -> Dict[str, Any]:
    """Devuelve diagnóstico útil de configuración y prueba de Groq."""
    key = GROQ_API_KEY or ""
    masked_key = (key[:4] + "*" * max(0, len(key) - 8) + key[-4:]) if len(key) >= 8 else ("****" if key else "")
    prueba = probar_conexion_groq()
    return {
        "env_path_cargado": GROQ_ENV_PATH_CARGADO,
        "groq_api_base": GROQ_API_BASE,
        "groq_model": GROQ_MODEL,
        "groq_key_masked": masked_key,
        "groq_key_length": len(key),
        "groq_enabled": GROQ_ENABLED,
        "disabled_reason": GROQ_DISABLED_REASON,
        "last_error": GROQ_LAST_ERROR,
        "probe_result": prueba,
    }


def extraer_operacion_con_groq(mensaje: str) -> Optional[Dict[str, Any]]:
    """
    Extracción operativa con few-shot. Groq solo estructura datos;
    no decide stock, precios, pedidos ni movimientos.
    """
    if not GROQ_ENABLED:
        return None
    system_prompt = (
        "Eres un extractor operativo para un chatbot retail EcoMarket. "
        "Devuelve solo JSON válido. No expliques. No inventes productos, stock, precios ni códigos de pedido. "
        "Tu tarea es extraer intención, acción, cliente e items solicitados. "
        "El backend validará todo contra PostgreSQL. "
        "Contrato JSON: {\"intencion\": string, \"accion\": string, "
        "\"cliente\": {\"nombre\": string|null, \"email\": string|null, \"telefono\": string|null}, "
        "\"items\": [{\"producto\": string, \"cantidad\": integer}]}. "
        "acciones permitidas: agregar_carrito, finalizar_compra, ver_carrito, vaciar_carrito, consultar, otro."
    )
    few_shot = [
        {"role": "user", "content": "hola, quiero comprar 5 manzanas y 3 leches enteras"},
        {"role": "assistant", "content": json.dumps({"intencion": "Compra de productos", "accion": "agregar_carrito", "cliente": {"nombre": None, "email": None, "telefono": None}, "items": [{"producto": "manzanas", "cantidad": 5}, {"producto": "leche entera", "cantidad": 3}]}, ensure_ascii=False)},
        {"role": "user", "content": "Soy Ana López, mi correo es ana.lopez@consultoria.es y mi teléfono 677889900. Finaliza la compra"},
        {"role": "assistant", "content": json.dumps({"intencion": "Compra de productos", "accion": "finalizar_compra", "cliente": {"nombre": "Ana López", "email": "ana.lopez@consultoria.es", "telefono": "677889900"}, "items": []}, ensure_ascii=False)},
        {"role": "user", "content": "agrega dos panes sin gluten, una mantequilla y 6 huevos"},
        {"role": "assistant", "content": json.dumps({"intencion": "Compra de productos", "accion": "agregar_carrito", "cliente": {"nombre": None, "email": None, "telefono": None}, "items": [{"producto": "pan sin gluten", "cantidad": 2}, {"producto": "mantequilla", "cantidad": 1}, {"producto": "huevos", "cantidad": 6}]}, ensure_ascii=False)},
    ]
    content = _llamar_groq_chat(
        [{"role": "system", "content": system_prompt}, *few_shot, {"role": "user", "content": mensaje}],
        max_completion_tokens=450,
        temperature=GROQ_TEMPERATURE_EXTRACTION,
        feature_name="Groq extracción operativa",
    )
    if not content:
        return None
    return validar_extraccion_operativa(_extraer_json_groq(content))


def resolver_items_extraidos(items_extraidos: List[Dict[str, Any]], service_obj) -> List[Dict[str, Any]]:
    """Convierte productos textuales de Groq a códigos del catálogo validado."""
    resueltos = []
    vistos = set()
    for item in items_extraidos:
        codigo = service_obj.buscar_producto(str(item.get("producto") or ""))
        if not codigo:
            continue
        cantidad = int(item.get("cantidad") or CANTIDAD_POR_DEFECTO)
        if cantidad <= 0:
            continue
        if codigo in vistos:
            for existente in resueltos:
                if existente["codigo_producto"] == codigo:
                    existente["cantidad"] += cantidad
                    break
            continue
        vistos.add(codigo)
        resueltos.append({"codigo_producto": codigo, "cantidad": cantidad, "posicion": len(resueltos)})
    return resueltos


UNIDADES_TEXTO = {
    "un": 1, "una": 1, "uno": 1,
    "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5,
    "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10,
    "once": 11, "doce": 12, "trece": 13, "catorce": 14, "quince": 15,
    "dieciseis": 16, "dieciséis": 16, "diecisiete": 17, "dieciocho": 18, "diecinueve": 19,
    "veinte": 20,
}


def _cantidad_en_segmento(segmento_n: str, alias_start: int) -> int:
    """Busca la cantidad más cercana antes del producto dentro del mismo segmento."""
    prefijo = segmento_n[:alias_start].strip()
    if not prefijo:
        return CANTIDAD_POR_DEFECTO

    tokens = re.findall(r"\b\d+\b|\b[a-záéíóúüñ]+\b", prefijo)
    for token in reversed(tokens):
        if token.isdigit():
            cantidad = int(token)
            return cantidad if cantidad > 0 else CANTIDAD_POR_DEFECTO
        if token in UNIDADES_TEXTO:
            return UNIDADES_TEXTO[token]
    return CANTIDAD_POR_DEFECTO


def _segmentar_items_compra(mensaje: str, service_obj) -> List[Dict[str, Any]]:
    """
    Divide el mensaje en segmentos de compra sin que una cantidad contamine otro producto.
    Ejemplo: '4 manzanas, 8 tomates y un detergente' => 3 segmentos.
    """
    msg_n = _normalizar(mensaje)
    msg_n = re.sub(
        r"\b(?:hola|buenos dias|buenas tardes|buenas noches|deseo|quiero|necesito|comprar|agregar|anadir|añadir|me llevo|por favor|dame|darme|ponme|agregame|agrégame|sumame|súmame)\b",
        " ",
        msg_n,
    )
    msg_n = re.sub(r"\s+", " ", msg_n).strip()

    segmentos = []
    patron = re.compile(
        r"(?:(?P<cantidad_num>\d+)|(?P<cantidad_txt>un|una|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|diez|once|doce|trece|catorce|quince|dieciseis|dieciséis|diecisiete|dieciocho|diecinueve|veinte))\s+"
        r"(?P<producto>.*?)(?=,|\s+y\s+(?:\d+|un|una|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|diez|once|doce|trece|catorce|quince|dieciseis|dieciséis|diecisiete|dieciocho|diecinueve|veinte)\s+|$)"
    )

    for match in patron.finditer(msg_n):
        cantidad = int(match.group("cantidad_num")) if match.group("cantidad_num") else UNIDADES_TEXTO.get(match.group("cantidad_txt"), CANTIDAD_POR_DEFECTO)
        producto_txt = re.sub(r"^\s*y\s+", "", match.group("producto")).strip(" ,.;")
        if producto_txt and cantidad > 0:
            segmentos.append({"texto": producto_txt, "cantidad": cantidad, "cantidad_explicita": True, "posicion": match.start()})

    cobertura = [False] * len(msg_n)
    for match in patron.finditer(msg_n):
        for pos in range(match.start(), match.end()):
            cobertura[pos] = True

    for codigo, datos in service_obj._catalogo.items():
        aliases = sorted(datos.get("aliases", []), key=len, reverse=True)
        for alias in aliases:
            alias_n = _normalizar(alias)
            if not alias_n:
                continue
            for match in re.finditer(re.escape(alias_n), msg_n):
                if any(cobertura[match.start():match.end()]):
                    continue
                segmentos.append({
                    "texto": match.group(0),
                    "cantidad": None,
                    "cantidad_explicita": False,
                    "posicion": match.start(),
                })
                break
            else:
                continue
            break

    return segmentos


def _resolver_segmentos_a_items(segmentos: List[Dict[str, Any]], service_obj) -> List[Dict[str, Any]]:
    items = []
    vistos = {}
    for segmento in segmentos:
        codigo = service_obj.buscar_producto(segmento["texto"])
        if not codigo:
            continue
        if not segmento.get("cantidad_explicita"):
            items.append({
                "codigo_producto": codigo,
                "cantidad": None,
                "cantidad_explicita": False,
                "posicion": int(segmento["posicion"]),
            })
            continue
        if codigo in vistos:
            items[vistos[codigo]]["cantidad"] += segmento["cantidad"]
            continue
        vistos[codigo] = len(items)
        items.append({
            "codigo_producto": codigo,
            "cantidad": int(segmento["cantidad"]),
            "cantidad_explicita": True,
            "posicion": int(segmento["posicion"]),
        })
    return items


def extraer_compra_condicional(mensaje: str) -> Dict[str, Any]:
    """
    Detecta cláusulas tipo:
    'si tienes yogurt entonces dame 2 unidades de pollo'
    y separa la parte base de la parte condicional.
    """
    texto = re.sub(r"\s+", " ", mensaje.strip())
    patron = re.compile(
        r"(?:^|,|\s+y\s+)?si\s+(?:tienes?|hay)\s+(?P<condicion>.+?)\s+entonces\s+(?P<accion>.*?)(?=(?:,|\s+y\s+)si\s+(?:tienes?|hay)\b|$)",
        flags=re.IGNORECASE,
    )

    clausulas = []
    partes_base = []
    last = 0

    for match in patron.finditer(texto):
        prefijo = texto[last:match.start()].strip(" ,")
        if prefijo:
            partes_base.append(prefijo)
        clausulas.append({
            "condicion": match.group("condicion").strip(" ,.;"),
            "accion": match.group("accion").strip(" ,.;"),
            "posicion": match.start(),
        })
        last = match.end()

    resto = texto[last:].strip(" ,")
    if resto:
        partes_base.append(resto)

    base_texto = " ".join(partes_base).strip(" ,")
    base_texto = re.sub(r"\s+(?:y|e)\s*$", "", base_texto, flags=re.IGNORECASE).strip(" ,")

    return {
        "base_texto": base_texto or mensaje,
        "clausulas": clausulas,
    }


def extraer_items_compra(mensaje: str, service_obj) -> List[Dict[str, Any]]:
    """
    Extrae varios productos y cantidades.
    Prioridad:
    1. Parser por segmentos con cantidades numéricas/textuales.
    2. Groq operativo si el parser local no resuelve suficientes productos.
    3. Fallback antiguo por aliases.
    """
    segmentos = _segmentar_items_compra(mensaje, service_obj)
    items_segmentados = _resolver_segmentos_a_items(segmentos, service_obj)
    if items_segmentados and len(items_segmentados) == len(segmentos):
        return sorted(items_segmentados, key=lambda item: item["posicion"])

    extraccion = extraer_operacion_con_groq(mensaje)
    if extraccion and extraccion.get("items"):
        items_groq = resolver_items_extraidos(extraccion["items"], service_obj)
        if len(items_groq) > len(items_segmentados):
            return items_groq

    msg_n = _normalizar(mensaje)
    items_por_codigo: Dict[str, Dict[str, Any]] = {}
    for codigo, datos in service_obj._catalogo.items():
        aliases = sorted(datos.get('aliases', []), key=len, reverse=True)
        mejor_match = None
        for alias in aliases:
            alias_n = _normalizar(alias)
            if not alias_n:
                continue
            match = re.search(re.escape(alias_n), msg_n)
            if match:
                mejor_match = match
                break
        if not mejor_match:
            continue
        cantidad = _cantidad_en_segmento(msg_n[max(0, mejor_match.start() - 45):mejor_match.end()], min(45, mejor_match.start()))
        items_por_codigo[codigo] = {'codigo_producto': codigo, 'cantidad': cantidad, 'posicion': mejor_match.start()}

    items_fallback = sorted(items_por_codigo.values(), key=lambda item: item['posicion'])
    return items_segmentados or items_fallback


In [ ]:
# ============================================================
# 4. RAG DOCUMENTAL
# Recupera conocimiento textual para politicas, pagos, promociones y FAQs.
# No sustituye SQL para inventario, clientes, pedidos o carrito.
# ============================================================

import math
from collections import Counter

RAG_ENABLED = True
RAG_TOP_K = 3
RAG_CANDIDATES = 5
RAG_MIN_SCORE = 0.08
RAG_DOCS_DIR = Path(os.getenv(
    "RAG_DOCS_DIR",
    r"C:\data_sciences\GIT\ChatBot_EcoMarket\ChatBot_EcoMarket\data\rag_knowledge"
))
RAG_INTENTIONS = {
    "Devoluciones y cambios",
    "Promociones y cupones",
    "Métodos de pago y compra",
}
RAG_SOURCE_HINTS = {
    "promociones": {"promociones_y_cupones.md", "faq_atencion_cliente.md"},
    "pagos": {"pagos_y_checkout.md", "faq_atencion_cliente.md"},
    "devoluciones": {"devoluciones_y_cambios.md", "faq_atencion_cliente.md"},
}

def build_prompt(context_chunks: list[dict], question: str) -> str:
    context = "\n\n".join(
        f"[Fuente: {chunk['source']} | score={chunk['score']:.3f}]\n{chunk['text']}"
        for chunk in context_chunks
    )
    return (
        "### Context\n"
        f"{context}\n\n"
        "### Question\n"
        f"{question}\n\n"
        "### Instructions\n"
        "- Responde solo con informacion presente en el contexto.\n"
        "- Si el contexto no basta, dilo claramente.\n"
        "- No inventes politicas, condiciones ni excepciones.\n"
        "- Responde en espanol, de forma clara y breve.\n"
    )


class RAGKnowledgeBase:
    def __init__(self, docs_dir: Path):
        self.docs_dir = docs_dir
        self.documents: list[dict] = []
        self.chunks: list[dict] = []
        self.vocab: dict[str, int] = {}
        self.idf: dict[str, float] = {}
        self.doc_vectors: list[dict[str, float]] = []

    def _tokenize(self, text: str) -> list[str]:
        text_n = _normalizar(text)
        return re.findall(r"\b[a-z0-9]{2,}\b", text_n)

    def _chunk_text(self, source: str, text: str) -> list[dict]:
        parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        chunks = []
        for idx, part in enumerate(parts):
            chunks.append({
                "id": f"{source}::chunk{idx+1}",
                "source": source,
                "text": part,
            })
        return chunks

    def load(self) -> None:
        if not self.docs_dir.exists():
            raise FileNotFoundError(f"No existe la carpeta documental RAG: {self.docs_dir}")

        self.documents = []
        self.chunks = []
        for path in sorted(self.docs_dir.glob("*.md")):
            text = path.read_text(encoding="utf-8")
            self.documents.append({"source": path.name, "text": text})
            self.chunks.extend(self._chunk_text(path.name, text))

        self._build_index()
        print(f"✓ RAG cargado: {len(self.documents)} documentos, {len(self.chunks)} chunks")

    def _build_index(self) -> None:
        tokenized = [self._tokenize(chunk["text"]) for chunk in self.chunks]
        df = Counter()
        for tokens in tokenized:
            for token in set(tokens):
                df[token] += 1

        total_docs = max(len(tokenized), 1)
        self.vocab = {token: idx for idx, token in enumerate(sorted(df))}
        self.idf = {
            token: math.log((1 + total_docs) / (1 + freq)) + 1.0
            for token, freq in df.items()
        }

        self.doc_vectors = [self._vectorize_tokens(tokens) for tokens in tokenized]

    def _vectorize_tokens(self, tokens: list[str]) -> dict[str, float]:
        if not tokens:
            return {}
        tf = Counter(tokens)
        total = sum(tf.values())
        vec = {}
        for token, freq in tf.items():
            vec[token] = (freq / total) * self.idf.get(token, 0.0)
        return vec

    def _cosine(self, a: dict[str, float], b: dict[str, float]) -> float:
        if not a or not b:
            return 0.0
        common = set(a).intersection(b)
        dot = sum(a[t] * b[t] for t in common)
        norm_a = math.sqrt(sum(v * v for v in a.values()))
        norm_b = math.sqrt(sum(v * v for v in b.values()))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    def _rerank_score(self, query_tokens: list[str], chunk_text: str, base_score: float) -> float:
        chunk_tokens = set(self._tokenize(chunk_text))
        if not chunk_tokens:
            return base_score
        overlap = len(set(query_tokens).intersection(chunk_tokens)) / max(len(set(query_tokens)), 1)
        return 0.7 * base_score + 0.3 * overlap

    def retrieve(self, question: str, top_k: int = RAG_TOP_K, candidates: int = RAG_CANDIDATES, allowed_sources: Optional[set[str]] = None) -> list[dict]:
        query_tokens = self._tokenize(question)
        query_vec = self._vectorize_tokens(query_tokens)

        scored = []
        for chunk, vec in zip(self.chunks, self.doc_vectors):
            if allowed_sources and chunk["source"] not in allowed_sources:
                continue
            score = self._cosine(query_vec, vec)
            scored.append({**chunk, "score": score})

        scored.sort(key=lambda x: x["score"], reverse=True)
        candidates_list = scored[:candidates]
        reranked = []
        for chunk in candidates_list:
            reranked.append({
                **chunk,
                "score": self._rerank_score(query_tokens, chunk["text"], chunk["score"]),
            })
        reranked.sort(key=lambda x: x["score"], reverse=True)
        return reranked[:top_k]


def _limpiar_chunk_text(text: str) -> str:
    text = re.sub(r"^#+\s*", "", text.strip())
    text = re.sub(r"\s+", " ", text)
    return text.strip(" -")


def _responder_rag_local(context_chunks: list[dict], fallback_text: str = "") -> str:
    partes = []
    vistos = set()
    for chunk in context_chunks:
        limpio = _limpiar_chunk_text(chunk.get("text", ""))
        if not limpio or len(limpio) < 20:
            continue
        if limpio.lower() in vistos:
            continue
        vistos.add(limpio.lower())
        partes.append(limpio)
    if not partes:
        return fallback_text or (
            "No tengo suficiente contexto documental para responder con seguridad. "
            "Puedo ayudarte con una consulta más concreta o derivarte a soporte."
        )
    if len(partes) == 1:
        return partes[0]
    return "Esto es lo que aplica en EcoMarket: " + " ".join(partes[:3])


def responder_rag_con_groq(question: str, context_chunks: list[dict], fallback_text: str = "") -> str:
    prompt = build_prompt(context_chunks, question)
    if not GROQ_ENABLED:
        return _responder_rag_local(context_chunks, fallback_text=fallback_text)

    content = _llamar_groq_chat(
        [
            {
                "role": "system",
                "content": (
                    "Eres un asistente de soporte retail. Responde solo con base en el contexto recibido. "
                    "Si la respuesta no esta claramente respaldada por el contexto, dilo sin inventar."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=350,
        temperature=GROQ_TEMPERATURE_RAG,
        feature_name="Groq RAG documental",
    )
    if not content:
        return _responder_rag_local(context_chunks, fallback_text=fallback_text)
    return content


def responder_pregunta_documental(question: str, fallback_text: str = "", allowed_sources: Optional[set[str]] = None) -> str:
    if not RAG_ENABLED:
        return fallback_text or "No tengo RAG habilitado en este momento."
    resultados = rag_kb.retrieve(question, allowed_sources=allowed_sources)
    if not resultados or resultados[0]["score"] < RAG_MIN_SCORE:
        return fallback_text or (
            "No tengo suficiente contexto documental para responder con seguridad. "
            "Puedo ayudarte con una consulta más concreta o derivarte a soporte."
        )
    return responder_rag_con_groq(question, resultados, fallback_text=fallback_text)


rag_kb = RAGKnowledgeBase(RAG_DOCS_DIR)
try:
    rag_kb.load()
except Exception as exc:
    RAG_ENABLED = False
    print(f"⚠ No se pudo inicializar RAG: {exc}")


## 5. Reglas de intención

In [ ]:
# ============================================================
# 5. REGLAS DE INTENCION + CLASIFICADOR ZERO-SHOT
# La detección por reglas tiene prioridad sobre el modelo.
# ============================================================

def detectar_intencion_por_reglas(mensaje: str) -> Optional[str]:
    msg_n = _normalizar(mensaje)

    if necesita_soporte_por_palabras_clave(mensaje):
        return INTENCION_SOPORTE

    if contiene_alguna(msg_n, ["caduc", "vencid", "mal estado", "danad", "roto", "podrid"]):
        return "Producto dañado caducado o en mal estado"

    if contiene_alguna(msg_n, [
        "no ha llegado", "no llego", "no recibi", "incompleto",
        "faltan", "falta", "equivocado", "entrega tarde", "retraso"
    ]):
        return "Incidencia con pedido incompleto o no recibido"

    if contiene_alguna(msg_n, ["devolver", "devolucion", "cambiar", "cambio", "reembolso"]):
        return "Devoluciones y cambios"

    if contiene_alguna(msg_n, ["cupon", "promocion", "descuento", "oferta", "puntos"]):
        return "Promociones y cupones"

    if contiene_alguna(msg_n, ["tarjeta", "pago", "pagar", "checkout", "paypal", "bizum"]):
        return "Métodos de pago y compra"

    if quiere_ver_categorias(mensaje) and not service.buscar_categoria(mensaje):
        return "Disponibilidad de productos"

    if service.buscar_categoria(mensaje) and contiene_alguna(msg_n, [
        "disponible", "stock", "inventario", "tienen", "hay",
        "categoria", "categorias", "productos"
    ]):
        return "Disponibilidad de productos"

    if es_solicitud_finalizar_compra(mensaje) or es_solicitud_ver_carrito(mensaje) or es_solicitud_vaciar_carrito(mensaje):
        return "Compra de productos"

    if service.buscar_producto(mensaje) and es_solicitud_compra(mensaje):
        return "Compra de productos"

    if service.buscar_producto(mensaje) and (
        contiene_alguna(msg_n, [
            "disponible", "stock", "inventario", "tienen", "hay",
            "comprar", "disponibilidad", "quiero", "necesito", "llevar", "pedir"
        ])
        or extraer_cantidad_solicitada(mensaje) > CANTIDAD_POR_DEFECTO
    ):
        return "Disponibilidad de productos"

    if extraer_id_pedido(mensaje) or contiene_alguna(msg_n, [
        "estado del pedido", "donde esta mi pedido", "seguimiento"
    ]):
        return "Estado del pedido"

    return None


# ── Carga del modelo zero-shot ────────────────────────────────────────────────
import os
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")

def cargar_clasificador_zero_shot():
    print("Cargando modelo NLP (mDeBERTa)... esto puede tardar unos segundos.")
    try:
        from transformers import pipeline as hf_pipeline
        clf = hf_pipeline(
            "zero-shot-classification",
            model=MODEL_NAME,
            tokenizer=MODEL_NAME,
            use_fast=False,
            local_files_only=MODEL_LOCAL_ONLY,
        )
        print("✓ Modelo cargado.")
        return clf
    except Exception as exc:
        print(f"⚠ No se pudo cargar el modelo: {exc}")
        print("  El chatbot usará solo reglas de intención.")
        return None

clasificador = cargar_clasificador_zero_shot()


def _extraer_json_groq(texto: str) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(texto)
    except Exception:
        match = re.search(r"\{.*\}", texto, flags=re.DOTALL)
        if not match:
            return None
        try:
            return json.loads(match.group(0))
        except Exception:
            return None


def interpretar_con_groq(mensaje: str) -> Optional[Dict[str, Any]]:
    """Usa Groq solo como fallback de interpretación, no como motor transaccional."""
    if not GROQ_ENABLED:
        return None

    system_prompt = (
        "Eres un clasificador de intención para un chatbot retail. Responde solo JSON válido. "
        "No inventes stock, pedidos ni datos de negocio. El backend decidirá operaciones. "
        "Intenciones permitidas: Compra de productos, Disponibilidad de productos, Estado del pedido, "
        "Incidencia con pedido incompleto o no recibido, Devoluciones y cambios, Producto dañado caducado o en mal estado, "
        "Promociones y cupones, Métodos de pago y compra, Hablar con soporte humano. "
        "Devuelve campos: intencion, confianza, producto, categoria, cantidad, accion. "
        "accion puede ser agregar_carrito, finalizar_compra, ver_carrito, vaciar_carrito, consultar, otro."
    )
    content = _llamar_groq_chat(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": mensaje},
        ],
        max_completion_tokens=250,
        temperature=GROQ_TEMPERATURE_INTENT,
        feature_name="Groq fallback de intención",
    )
    if not content:
        return None
    return _extraer_json_groq(content)


def procesar_mensaje(mensaje: str):
    """
    Determina la intención del usuario con cascada:
    reglas fuertes -> Groq exploratorio -> mDeBERTa alta confianza -> Groq fallback -> soporte.
    Devuelve: (intencion, score, ranking, origen)
    """
    intencion_regla = detectar_intencion_por_reglas(mensaje)
    if intencion_regla:
        return intencion_regla, 1.0, [(intencion_regla, 1.0)], "regla_fuerte"

    ranking = []
    intencion = INTENCION_SOPORTE
    score = 0.0

    if GROQ_ENABLED and es_consulta_exploratoria(mensaje):
        interpretacion = interpretar_con_groq(mensaje)
        if interpretacion and interpretacion.get("intencion") in ETIQUETAS_NEGOCIO:
            groq_score = float(interpretacion.get("confianza") or 0.78)
            return interpretacion["intencion"], groq_score, ranking, "groq_fallback_exploratorio"

    if clasificador is not None:
        resultado = clasificador(mensaje, ETIQUETAS_NEGOCIO)
        ranking = list(zip(resultado["labels"], resultado["scores"]))
        intencion = resultado["labels"][0]
        score = float(resultado["scores"][0])
        if score >= HIGH_CONFIDENCE_THRESHOLD:
            return intencion, score, ranking, "modelo_zero_shot_alta_confianza"

        if score < GROQ_PREFER_BELOW_SCORE:
            interpretacion = interpretar_con_groq(mensaje)
            if interpretacion and interpretacion.get("intencion") in ETIQUETAS_NEGOCIO:
                groq_score = float(interpretacion.get("confianza") or max(score, 0.72))
                return interpretacion["intencion"], groq_score, ranking, "groq_fallback"

    elif GROQ_ENABLED:
        interpretacion = interpretar_con_groq(mensaje)
        if interpretacion and interpretacion.get("intencion") in ETIQUETAS_NEGOCIO:
            groq_score = float(interpretacion.get("confianza") or 0.72)
            return interpretacion["intencion"], groq_score, ranking, "groq_fallback_sin_mdeberta"

    if clasificador is None:
        return INTENCION_SOPORTE, 0.0, [(INTENCION_SOPORTE, 0.0)], "modelo_no_disponible"

    if score < CONFIDENCE_THRESHOLD:
        return INTENCION_SOPORTE, score, ranking, "modelo_zero_shot_baja_confianza"

    return intencion, score, ranking, "modelo_zero_shot_media_confianza"




    ranking = []
    intencion = INTENCION_SOPORTE
    score = 0.0

    if clasificador is not None:
        resultado = clasificador(mensaje, ETIQUETAS_NEGOCIO)
        ranking = list(zip(resultado["labels"], resultado["scores"]))
        intencion = resultado["labels"][0]
        score = float(resultado["scores"][0])
        if score >= HIGH_CONFIDENCE_THRESHOLD:
            return intencion, score, ranking, "modelo_zero_shot_alta_confianza"

    interpretacion = interpretar_con_groq(mensaje)
    if interpretacion and interpretacion.get("intencion") in ETIQUETAS_NEGOCIO:
        groq_score = float(interpretacion.get("confianza") or max(score, 0.70))
        return interpretacion["intencion"], groq_score, ranking, "groq_fallback"

    if clasificador is None:
        return INTENCION_SOPORTE, 0.0, [(INTENCION_SOPORTE, 0.0)], "modelo_no_disponible"

    if score < CONFIDENCE_THRESHOLD:
        return INTENCION_SOPORTE, score, ranking, "modelo_zero_shot_baja_confianza"

    return intencion, score, ranking, "modelo_zero_shot_media_confianza"


Cargando modelo NLP (mDeBERTa)... esto puede tardar unos segundos.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo cargado.


## 6. Generación de respuestas

In [ ]:
# ============================================================
# 6. GENERACION DE RESPUESTAS + CARRITO
# Todas las funciones usan service.* — cero SQL directo aquí.
# ============================================================

@dataclass
class EstadoConversacion:
    esperando_id_pedido:               bool = False
    esperando_confirmacion_categorias: bool = False
    esperando_nombre_cliente:          bool = False
    esperando_metodo_pago:             bool = False
    esperando_confirmacion_pedido:     bool = False
    esperando_cantidad_producto:       bool = False
    esperando_confirmacion_contrapropuesta: bool = False
    ultima_intencion:                  Optional[str] = None
    carrito:                           List[Dict[str, Any]] = field(default_factory=list)
    cliente_pendiente:                 Optional[Dict[str, Any]] = None
    ultima_extraccion_operativa:       Optional[Dict[str, Any]] = None
    contrapropuesta_pendiente:         Optional[Dict[str, Any]] = None
    producto_pendiente_cantidad:       Optional[Dict[str, Any]] = None
    metodo_pago_pendiente:             Optional[str] = None


# ── Formateo de detalle, movimientos y carrito ───────────────────────────────

def _formatear_detalle(lineas: List[Dict]) -> str:
    if not lineas:
        return ""
    partes = []
    for ln in lineas:
        texto = (
            f'{ln["producto"]}: comprado {ln["cantidad_comprada"]}, '
            f'entregado {ln["cantidad_entregada"]}, estado {ln["estado_linea"]}'
        )
        if ln.get("motivo_incidencia"):
            texto += f' ({ln["motivo_incidencia"]})'
        partes.append(texto)
    return " Detalle: " + "; ".join(partes) + "."


def _formatear_movimientos(movimientos: List[Dict]) -> str:
    if not movimientos:
        return ""
    compras = sum(m["cantidad"] for m in movimientos if m["tipo_movimiento"] == "compra")
    devols  = sum(m["cantidad"] for m in movimientos if m["tipo_movimiento"] == "devolucion")
    partes  = []
    if compras:
        partes.append(f"compras registradas: {compras} unidades")
    if devols:
        partes.append(f"devoluciones registradas: {devols} unidades")
    return (" Movimientos de inventario: " + ", ".join(partes) + ".") if partes else ""


def _calcular_resumen_comercial(estado: EstadoConversacion) -> Dict[str, Any]:
    lineas = []
    subtotal_bruto = 0.0
    descuento_volumen = 0.0

    for item in estado.carrito:
        cantidad = int(item["cantidad"])
        precio = float(item.get("precio", 0))
        subtotal_linea = round(precio * cantidad, 2)
        descuento_linea = 0.0
        if cantidad >= UMBRAL_DESCUENTO_VOLUMEN:
            descuento_linea = round(subtotal_linea * PORCENTAJE_DESCUENTO_VOLUMEN, 2)
        subtotal_bruto += subtotal_linea
        descuento_volumen += descuento_linea
        lineas.append({
            "codigo_producto": item["codigo_producto"],
            "nombre_producto": item["nombre_producto"],
            "cantidad": cantidad,
            "precio": precio,
            "subtotal_bruto": subtotal_linea,
            "descuento_volumen": descuento_linea,
            "subtotal_neto": round(subtotal_linea - descuento_linea, 2),
        })

    subtotal_tras_volumen = round(subtotal_bruto - descuento_volumen, 2)
    descuento_carrito = round(subtotal_tras_volumen * PORCENTAJE_DESCUENTO_CARRITO, 2) if subtotal_tras_volumen >= UMBRAL_DESCUENTO_CARRITO else 0.0
    total_final = round(subtotal_tras_volumen - descuento_carrito, 2)
    envio_gratis = subtotal_tras_volumen >= UMBRAL_ENVIO_GRATIS

    beneficios = []
    productos_con_volumen = [l for l in lineas if l["descuento_volumen"] > 0]
    for linea in productos_con_volumen:
        beneficios.append(
            f"5% por volumen en {linea['nombre_producto']} (-{linea['descuento_volumen']:.2f} €)"
        )
    if envio_gratis:
        beneficios.append("envío gratis")
    if descuento_carrito > 0:
        beneficios.append(f"10% de descuento sobre el carrito (-{descuento_carrito:.2f} €)")

    return {
        "lineas": lineas,
        "subtotal_bruto": round(subtotal_bruto, 2),
        "descuento_volumen": round(descuento_volumen, 2),
        "subtotal_tras_volumen": subtotal_tras_volumen,
        "descuento_carrito": descuento_carrito,
        "envio_gratis": envio_gratis,
        "total_final": total_final,
        "beneficios": beneficios,
    }


def _total_carrito(estado: EstadoConversacion) -> float:
    return _calcular_resumen_comercial(estado)["total_final"]


def _items_carrito_formateados(estado: EstadoConversacion) -> str:
    if not estado.carrito:
        return ""
    resumen = _calcular_resumen_comercial(estado)
    partes = []
    for item in resumen["lineas"]:
        cantidad = int(item["cantidad"])
        unidad = "unidad" if cantidad == 1 else "unidades"
        partes.append(f"- {cantidad} {unidad} de {item['nombre_producto']} ({item['subtotal_neto']:.2f} €)")
    return "\n".join(partes)


def _resumen_carrito(estado: EstadoConversacion) -> str:
    if not estado.carrito:
        return "Tu carrito está vacío."

    resumen = _calcular_resumen_comercial(estado)
    partes = []
    for item in resumen["lineas"]:
        cantidad = int(item["cantidad"])
        unidad = "unidad" if cantidad == 1 else "unidades"
        partes.append(f"{cantidad} {unidad} de {item['nombre_producto']} ({item['subtotal_neto']:.2f} €)")

    extras = []
    if resumen["beneficios"]:
        extras.append("Beneficios aplicados: " + "; ".join(resumen["beneficios"]) + ".")
    extras.append(f"Total estimado: {resumen['total_final']:.2f} €.")
    return "Tu carrito tiene: " + "; ".join(partes) + ". " + " ".join(extras)


def _limpiar_checkout(estado: EstadoConversacion) -> None:
    estado.esperando_nombre_cliente = False
    estado.esperando_metodo_pago = False
    estado.esperando_confirmacion_pedido = False
    estado.cliente_pendiente = None
    estado.metodo_pago_pendiente = None


def _vaciar_carrito(estado: EstadoConversacion) -> str:
    estado.carrito.clear()
    estado.contrapropuesta_pendiente = None
    estado.esperando_confirmacion_contrapropuesta = False
    estado.esperando_cantidad_producto = False
    estado.producto_pendiente_cantidad = None
    _limpiar_checkout(estado)
    return "Listo, vacié tu carrito. Podemos empezar una compra nueva cuando quieras."


def _agregar_item_carrito(estado: EstadoConversacion, codigo_producto: str, cantidad: int) -> None:
    datos = service._catalogo[codigo_producto]
    for item in estado.carrito:
        if item["codigo_producto"] == codigo_producto:
            item["cantidad"] += cantidad
            return
    estado.carrito.append({
        "codigo_producto": codigo_producto,
        "nombre_producto": datos["nombre"],
        "cantidad": cantidad,
        "precio": datos.get("precio", 0),
    })


def _quitar_contexto_contrapropuesta(estado: EstadoConversacion) -> None:
    estado.esperando_confirmacion_contrapropuesta = False
    estado.contrapropuesta_pendiente = None


def _pedir_cantidad_producto(codigo_producto: str, estado: EstadoConversacion) -> str:
    datos = service._catalogo.get(codigo_producto)
    if not datos:
        return "No encontré información de ese producto en el sistema."
    estado.esperando_cantidad_producto = True
    estado.producto_pendiente_cantidad = {"codigo_producto": codigo_producto, "nombre_producto": datos["nombre"]}
    return f"¿Cuántas unidades de {datos['nombre']} deseas agregar al carrito?"


def responder_cantidad_producto_pendiente(mensaje: str, estado: EstadoConversacion) -> str:
    cantidad = extraer_cantidad_solicitada(mensaje)
    pendiente = estado.producto_pendiente_cantidad or {}
    codigo = pendiente.get("codigo_producto")
    if not codigo:
        estado.esperando_cantidad_producto = False
        return "No tengo un producto pendiente. Indícame qué producto quieres comprar."
    if not tiene_cantidad_explicita(mensaje):
        return f"Indícame la cantidad en número para {pendiente.get('nombre_producto')}, por ejemplo: 2."
    estado.esperando_cantidad_producto = False
    estado.producto_pendiente_cantidad = None
    return agregar_producto_al_carrito(codigo, cantidad, estado)


def _tokens_relevantes(texto: str) -> List[str]:
    stopwords = {
        "de", "la", "el", "los", "las", "un", "una", "unos", "unas", "y", "o",
        "quiero", "necesito", "comprar", "agregar", "anadir", "añadir", "llevar",
        "producto", "productos", "categoria", "categorias", "tengo", "tienes", "hay"
    }
    tokens = re.findall(r"\b[a-z0-9]{2,}\b", _normalizar(texto))
    return [t for t in tokens if t not in stopwords]


def _alternativas_por_categoria_de_codigo(codigo_producto: str, limite: int = 3) -> List[Dict[str, Any]]:
    base = service._catalogo.get(codigo_producto)
    if not base:
        return []
    categoria = base.get("categoria")
    candidatos = []
    for codigo, datos in service._catalogo.items():
        if codigo == codigo_producto:
            continue
        if int(datos.get("stock", 0)) <= 0:
            continue
        score = 0
        if datos.get("categoria") == categoria:
            score += 3
        base_tokens = set(_tokens_relevantes(base.get("nombre", "")))
        cand_tokens = set(_tokens_relevantes(" ".join(datos.get("aliases", []))))
        score += len(base_tokens & cand_tokens)
        if score > 0:
            candidatos.append({"codigo_producto": codigo, "score": score, **datos})
    candidatos.sort(key=lambda x: (x["score"], x.get("stock", 0), -x.get("precio", 0)), reverse=True)
    return candidatos[:limite]


def _alternativas_por_texto(texto: str, limite: int = 3) -> List[Dict[str, Any]]:
    query_tokens = set(_tokens_relevantes(texto))
    if not query_tokens:
        return []
    candidatos = []
    for codigo, datos in service._catalogo.items():
        if int(datos.get("stock", 0)) <= 0:
            continue
        alias_tokens = set()
        for alias in datos.get("aliases", []):
            alias_tokens.update(_tokens_relevantes(alias))
        nombre_tokens = set(_tokens_relevantes(datos.get("nombre", "")))
        overlap = len(query_tokens & alias_tokens)
        if overlap == 0 and not (query_tokens & nombre_tokens):
            continue
        score = overlap * 3 + len(query_tokens & nombre_tokens)
        categoria_tokens = set(_tokens_relevantes(datos.get("categoria", "")))
        score += len(query_tokens & categoria_tokens)
        candidatos.append({"codigo_producto": codigo, "score": score, **datos})
    candidatos.sort(key=lambda x: (x["score"], x.get("stock", 0), -x.get("precio", 0)), reverse=True)
    return candidatos[:limite]


def _mensaje_alternativas(alternativas: List[Dict[str, Any]], prefijo: str) -> str:
    if not alternativas:
        return prefijo
    lista = ", ".join(a["nombre"] for a in alternativas)
    return prefijo + f" Puedo ofrecerte estas alternativas: {lista}. ¿Quieres que agregue alguna al carrito?"


def _extraer_metodo_pago(mensaje: str) -> Optional[str]:
    msg_n = _normalizar(mensaje)
    if "contra entrega" in msg_n or "contraentrega" in msg_n:
        return "contra_entrega"
    if "efectivo" in msg_n:
        return "efectivo"
    if any(t in msg_n for t in ["tarjeta", "credito", "crédito", "debito", "débito"]):
        return "tarjeta"
    return None


def _etiqueta_metodo_pago(metodo_pago: Optional[str]) -> str:
    etiquetas = {
        "tarjeta": "tarjeta",
        "efectivo": "efectivo",
        "contra_entrega": "contra entrega",
    }
    return etiquetas.get(metodo_pago or "", "sin definir")


def _mensaje_pago_operativo(metodo_pago: str) -> str:
    if metodo_pago == "efectivo":
        return "Por favor, acércate a la tienda para realizar tu pago en efectivo."
    if metodo_pago == "tarjeta":
        return "Por favor, acércate a la tienda para realizar tu pago con tarjeta."
    if metodo_pago == "contra_entrega":
        return "El pago quedará registrado contra entrega y se cobrará al momento de entregar tu pedido."
    return ""


# ── Respuestas por intención ──────────────────────────────────────────────────

def responder_estado_pedido(mensaje: str, estado: EstadoConversacion) -> str:
    codigo = extraer_id_pedido(mensaje)

    if not codigo:
        estado.esperando_id_pedido = True
        return "Claro, puedo revisar tu pedido. Indícame la referencia, por ejemplo PED-123."

    estado.esperando_id_pedido = False
    pedido = service.consultar_pedido(codigo)

    if not pedido:
        return f"No encuentro el pedido {codigo}. Revisa la referencia o te derivo con soporte."

    detalle = _formatear_detalle(service.consultar_detalle_pedido(codigo))
    movs = _formatear_movimientos(service.consultar_movimientos_pedido(codigo))

    return (
        f"Tu pedido {codigo} está en estado: {pedido['estado_pedido']}. "
        f"Entrega estimada: {pedido['fecha_entrega_estimada']}."
        f"{detalle}{movs}"
    )


def responder_incidencia_pedido(mensaje: str) -> str:
    codigo = extraer_id_pedido(mensaje)
    if codigo and service.consultar_pedido(codigo):
        detalle = _formatear_detalle(service.consultar_detalle_pedido(codigo))
        return (
            f"Lamento la incidencia con el pedido {codigo}.{detalle} "
            "Si hay productos faltantes o en mal estado, puedo ayudarte a preparar la reclamación."
        )
    if codigo:
        return (
            f"Lamento la incidencia con el pedido {codigo}. "
            "Dime si faltan productos, llegó tarde o hubo un error en la entrega."
        )
    return (
        "Lamento lo ocurrido con tu pedido. Indícame el número de pedido "
        "y si no llegó, llegó incompleto o contiene productos equivocados."
    )


def responder_disponibilidad(mensaje: str, estado: EstadoConversacion) -> str:
    if quiere_ver_categorias(mensaje) and not service.buscar_categoria(mensaje):
        estado.esperando_confirmacion_categorias = False
        cats = service.listar_categorias()
        if not cats:
            return "No tengo categorías disponibles en este momento."
        return "Estas son las categorías disponibles: " + ", ".join(cats) + "."

    categoria = service.buscar_categoria(mensaje)
    if categoria:
        estado.esperando_confirmacion_categorias = True
        prods = service.consultar_productos_por_categoria(categoria)
        disponibles = [p for p in prods if p["disponible"]]
        if not disponibles:
            return f"Ahora mismo no tengo productos disponibles en la categoría {categoria}."
        lista = ", ".join(f'{p["nombre"]}' for p in disponibles)
        return (
            f"En la categoría {categoria} tengo disponibles: {lista}. "
            "¿Deseas conocer otras categorías o productos?"
        )

    codigo = service.buscar_producto(mensaje)
    if not codigo:
        alternativas = _alternativas_por_texto(mensaje)
        if alternativas:
            return _mensaje_alternativas(
                alternativas,
                "No tengo ese producto exactamente en el catálogo."
            )
        return (
            "Puedo consultar disponibilidad, pero necesito el producto exacto o la categoría. "
            "Por ejemplo: manzanas, pan sin gluten o categoría lácteos."
        )

    datos = service._catalogo.get(codigo)
    if not datos:
        return "No encontré información de ese producto en el sistema."

    stock = int(datos["stock"])
    nombre = datos["nombre"]
    cantidad = extraer_cantidad_solicitada(mensaje)

    if stock <= 0:
        alternativas = _alternativas_por_categoria_de_codigo(codigo)
        return _mensaje_alternativas(
            alternativas,
            f"Ahora mismo {nombre} aparece agotado."
        )

    if cantidad > stock:
        estado.contrapropuesta_pendiente = {"codigo_producto": codigo, "cantidad": stock}
        estado.esperando_confirmacion_contrapropuesta = True
        return (
            f"Puedo ayudarte con {nombre}, pero no alcanzo a cubrir {cantidad} unidades. "
            f"Ahora mismo puedo ofrecerte {stock}. ¿Quieres que agregue esa cantidad al carrito?"
        )

    if tiene_cantidad_explicita(mensaje) and es_solicitud_compra(mensaje):
        return agregar_producto_al_carrito(codigo, cantidad, estado)

    return (
        f"Sí, tenemos disponibilidad de {nombre}. "
        "¿Cuántas unidades deseas agregar al carrito?"
    )


def agregar_producto_al_carrito(codigo_producto: str, cantidad: int, estado: EstadoConversacion) -> str:
    datos = service._catalogo.get(codigo_producto)
    if not datos:
        return "No encontré información de ese producto en el sistema."
    if cantidad <= 0:
        return "Indícame una cantidad mayor a cero para poder agregarla al carrito."

    stock = int(datos["stock"])
    nombre = datos["nombre"]
    cantidad_en_carrito = sum(i["cantidad"] for i in estado.carrito if i["codigo_producto"] == codigo_producto)
    cantidad_total = cantidad_en_carrito + cantidad

    if stock <= 0:
        alternativas = _alternativas_por_categoria_de_codigo(codigo_producto)
        return _mensaje_alternativas(
            alternativas,
            f"Ahora mismo {nombre} aparece agotado."
        )

    if cantidad_total > stock:
        disponible_para_agregar = max(stock - cantidad_en_carrito, 0)
        estado.contrapropuesta_pendiente = {
            "codigo_producto": codigo_producto,
            "cantidad": disponible_para_agregar,
        }
        estado.esperando_confirmacion_contrapropuesta = disponible_para_agregar > 0
        if disponible_para_agregar <= 0:
            return (
                f"Ya tienes en el carrito la cantidad máxima disponible de {nombre}. "
                "Puedes finalizar la compra o revisar otros productos."
            )
        return (
            f"Para {nombre}, con lo que ya tienes en el carrito, puedo agregarte {disponible_para_agregar} unidades más. "
            "¿Quieres que agregue esa cantidad?"
        )

    _agregar_item_carrito(estado, codigo_producto, cantidad)
    _quitar_contexto_contrapropuesta(estado)
    unidad = "unidad" if cantidad == 1 else "unidades"
    return (
        f"Perfecto, agregué {cantidad} {unidad} de {nombre} al carrito. "
        f"{_resumen_carrito(estado)} ¿Deseas añadir otro producto o finalizar la compra?"
    )


def responder_confirmacion_contrapropuesta(mensaje: str, estado: EstadoConversacion) -> str:
    if es_respuesta_negativa(mensaje):
        _quitar_contexto_contrapropuesta(estado)
        return "Entendido, no lo agregué. Puedes pedirme otro producto o revisar tu carrito."
    if not es_respuesta_afirmativa(mensaje):
        return "Confírmame con 'sí' si quieres que agregue la cantidad propuesta al carrito, o 'no' para descartarla."

    propuesta = estado.contrapropuesta_pendiente or {}
    codigo = propuesta.get("codigo_producto")
    cantidad = int(propuesta.get("cantidad") or 0)
    _quitar_contexto_contrapropuesta(estado)
    if not codigo or cantidad <= 0:
        return "No tengo una contrapropuesta activa. Indícame el producto y la cantidad que deseas."
    return agregar_producto_al_carrito(codigo, cantidad, estado)


def iniciar_checkout(estado: EstadoConversacion) -> str:
    if not estado.carrito:
        return "Tu carrito está vacío. Dime qué producto quieres comprar y cuántas unidades deseas."
    estado.esperando_nombre_cliente = True
    estado.esperando_metodo_pago = False
    estado.esperando_confirmacion_pedido = False
    return (
        f"{_resumen_carrito(estado)} Para generar el pedido, indícame tu nombre completo, email y teléfono. "
        "Por ejemplo: María Gómez, maria@email.com, 600123456."
    )


def _fusionar_contacto(local: Dict[str, Optional[str]], groq_data: Optional[Dict[str, Any]]) -> Dict[str, Optional[str]]:
    cliente_groq = (groq_data or {}).get("cliente") if isinstance(groq_data, dict) else {}
    cliente_groq = cliente_groq if isinstance(cliente_groq, dict) else {}
    return {
        "nombre": local.get("nombre") or cliente_groq.get("nombre"),
        "email": local.get("email") or cliente_groq.get("email"),
        "telefono": local.get("telefono") or cliente_groq.get("telefono"),
    }


def responder_nombre_cliente_para_checkout(mensaje: str, estado: EstadoConversacion) -> str:
    contacto_local = extraer_contacto_local(mensaje)
    extraccion = extraer_operacion_con_groq(mensaje)
    contacto = _fusionar_contacto(contacto_local, extraccion)

    if not contacto.get("nombre"):
        return "Para crear el pedido necesito al menos tu nombre y apellido. Si puedes, incluye también email y teléfono."

    cliente = None
    if contacto.get("email"):
        cliente = None
    else:
        cliente = service.buscar_cliente_por_nombre(contacto["nombre"])

    estado.cliente_pendiente = {
        "nombre_cliente": cliente.get("nombre") if cliente else contacto["nombre"],
        "email": contacto.get("email"),
        "telefono": contacto.get("telefono"),
        "codigo_cliente": cliente["codigo_cliente"] if cliente else None,
        "cliente_existente": bool(cliente),
    }
    estado.esperando_nombre_cliente = False
    estado.esperando_metodo_pago = True

    accion_cliente = (
        f"encontré el cliente {cliente['codigo_cliente']}" if cliente
        else "crearé o actualizaré el cliente con los datos entregados al confirmar"
    )
    contacto_txt = []
    if estado.cliente_pendiente.get("email"):
        contacto_txt.append(f"email {estado.cliente_pendiente['email']}")
    if estado.cliente_pendiente.get("telefono"):
        contacto_txt.append(f"teléfono {estado.cliente_pendiente['telefono']}")
    contacto_msg = " Datos de contacto: " + ", ".join(contacto_txt) + "." if contacto_txt else ""
    return (
        f"Perfecto, {accion_cliente} para {estado.cliente_pendiente['nombre_cliente']}.{contacto_msg} "
        f"{_resumen_carrito(estado)} Ahora indícame el método de pago: tarjeta, efectivo o contra entrega."
    )


def responder_metodo_pago_checkout(mensaje: str, estado: EstadoConversacion) -> str:
    metodo = _extraer_metodo_pago(mensaje)
    if not metodo:
        return "Indícame un método de pago válido: tarjeta, efectivo o contra entrega."

    estado.metodo_pago_pendiente = metodo
    estado.esperando_metodo_pago = False
    estado.esperando_confirmacion_pedido = True

    instruccion = _mensaje_pago_operativo(metodo)
    return (
        f"Perfecto, dejaré tu pedido con pago {_etiqueta_metodo_pago(metodo)}. {instruccion} "
        f"{_resumen_carrito(estado)} Confírmame con 'sí' para generar el pedido o 'no' para cancelarlo."
    )


def _observaciones_checkout(estado: EstadoConversacion) -> str:
    resumen = _calcular_resumen_comercial(estado)
    partes = ["Pedido multiartículo creado desde conversación del chatbot"]
    if resumen["beneficios"]:
        partes.append("Promociones aplicadas: " + "; ".join(resumen["beneficios"]))
    if estado.metodo_pago_pendiente:
        partes.append("Método de pago: " + _etiqueta_metodo_pago(estado.metodo_pago_pendiente))
    return ". ".join(partes)


def confirmar_checkout(mensaje: str, estado: EstadoConversacion) -> str:
    if es_respuesta_negativa(mensaje):
        _limpiar_checkout(estado)
        return "Listo, dejé el carrito sin generar pedido. Puedes seguir agregando productos o vaciarlo."

    if not es_respuesta_afirmativa(mensaje):
        return "Confírmame con 'sí' para generar el pedido o 'no' para cancelarlo."

    if not estado.carrito:
        _limpiar_checkout(estado)
        return "Tu carrito está vacío. No generé ningún pedido."

    cliente = estado.cliente_pendiente or {}
    if not cliente.get("codigo_cliente"):
        cliente_db = service.obtener_o_crear_cliente_por_contacto(
            nombre=cliente.get("nombre_cliente"),
            email=cliente.get("email"),
            telefono=cliente.get("telefono"),
        )
        cliente["codigo_cliente"] = cliente_db["codigo_cliente"]

    items_confirmados = list(estado.carrito)
    resumen = _calcular_resumen_comercial(estado)
    metodo_pago = estado.metodo_pago_pendiente or None
    resultado = service.crear_pedido(
        codigo_cliente=cliente["codigo_cliente"],
        carrito=[{"codigo_producto": i["codigo_producto"], "cantidad": i["cantidad"]} for i in items_confirmados],
        canal="chatbot",
        metodo_pago=metodo_pago,
        observaciones=_observaciones_checkout(estado),
    )
    if not resultado["ok"]:
        _limpiar_checkout(estado)
        return "No pude crear el pedido: " + "; ".join(resultado["errores"])

    codigo_pedido = resultado["codigo_pedido"]
    estado.carrito.clear()
    _limpiar_checkout(estado)

    beneficios_txt = ""
    if resumen["beneficios"]:
        beneficios_txt = "\nBeneficios aplicados:\n- " + "\n- ".join(resumen["beneficios"]) + "\n"

    instruccion_pago = _mensaje_pago_operativo(metodo_pago) if metodo_pago else ""
    return (
        "Muchas gracias por comprar en EcoMarket.\n\n"
        f"Tu número de pedido es: {codigo_pedido}\n"
        f"Método de pago: {_etiqueta_metodo_pago(metodo_pago)}\n\n"
        "Items:\n" + _items_carrito_formateados(EstadoConversacion(carrito=items_confirmados)) + "\n"
        f"{beneficios_txt}\n"
        f"Importe total del pedido: {resumen['total_final']:.2f} €.\n"
        + (f"{instruccion_pago}" if instruccion_pago else "")
    ).strip()



def responder_promociones_mvp() -> str:
    return (
        "Sí. En EcoMarket tenemos estas promociones automáticas activas para el MVP: "
        "si compras 3 o más unidades del mismo producto, se aplica un 5 % de descuento por volumen sobre esa línea. "
        "Además, si el total del carrito supera 50 €, el pedido obtiene envío gratis. "
        "Y si el total promocional del carrito alcanza 75 € o más, se aplica un 10 % de descuento adicional sobre el total del carrito."
    )


def responder_producto_mal_estado(mensaje: str) -> str:
    codigo = service.buscar_producto(mensaje)
    if codigo:
        nombre = service._catalogo[codigo]["nombre"]
        return (
            f"Siento que hayas recibido {nombre} en mal estado. "
            "Conserva el ticket o número de pedido y, si puedes, una foto. "
            "Te derivo con atención al cliente."
        )
    return (
        "Siento que hayas recibido un producto en mal estado. "
        "Indícame el producto, el número de pedido y si tienes foto o ticket."
    )


def _procesar_items_compra(items: List[Dict[str, Any]], estado: EstadoConversacion, incluir_resumen: bool = True) -> Optional[Dict[str, Any]]:
    if not items:
        return None

    respuestas = []
    for item in sorted(items, key=lambda x: x["posicion"]):
        if item.get("cantidad") is None:
            resumen_parcial = (" ".join(respuestas) + " ") if respuestas else ""
            return {
                "response": (resumen_parcial + _pedir_cantidad_producto(item["codigo_producto"], estado)).strip(),
                "interrumpido": True,
                "messages": respuestas,
            }

        respuesta = agregar_producto_al_carrito(item["codigo_producto"], int(item["cantidad"]), estado)
        if estado.esperando_confirmacion_contrapropuesta or estado.esperando_cantidad_producto:
            return {
                "response": ((" ".join(respuestas) + " " + respuesta).strip()),
                "interrumpido": True,
                "messages": respuestas,
            }
        respuestas.append(respuesta.split(" Tu carrito tiene:")[0].strip())

    texto = " ".join(respuestas).strip()
    if incluir_resumen and texto:
        texto = f"{texto} {_resumen_carrito(estado)} ¿Deseas añadir otro producto o finalizar la compra?"
    return {"response": texto, "interrumpido": False, "messages": respuestas}



def responder_compra(mensaje: str, estado: EstadoConversacion) -> str:
    if es_solicitud_vaciar_carrito(mensaje):
        return _vaciar_carrito(estado)
    if es_solicitud_ver_carrito(mensaje):
        return _resumen_carrito(estado)
    if es_solicitud_finalizar_compra(mensaje):
        return iniciar_checkout(estado)

    extraccion = extraer_operacion_con_groq(mensaje)
    if extraccion:
        estado.ultima_extraccion_operativa = extraccion

    compra_condicional = extraer_compra_condicional(mensaje)
    base_texto = compra_condicional.get("base_texto") or mensaje
    clausulas = compra_condicional.get("clausulas") or []

    respuestas = []
    hubo_agregados = False

    items_base = extraer_items_compra(base_texto, service) if base_texto else []
    if items_base:
        resultado_base = _procesar_items_compra(items_base, estado, incluir_resumen=False)
        if resultado_base and resultado_base["interrumpido"]:
            return resultado_base["response"]
        if resultado_base:
            respuestas.extend(resultado_base["messages"])
            hubo_agregados = bool(resultado_base["messages"])

    if not items_base and not clausulas:
        if es_consulta_exploratoria(mensaje):
            categorias = service.listar_categorias()
            if categorias:
                return (
                    "Claro, puedo ayudarte a organizar la compra. "
                    "Puedo orientarte por categorías como: " + ", ".join(categorias) + ". "
                    "Si quieres, dime por ejemplo si buscas desayuno, fruta, limpieza o una compra para toda la semana."
                )
            return (
                "Claro, puedo ayudarte a organizar la compra. "
                "Cuéntame si buscas desayuno, fruta, limpieza o una compra semanal y te propongo opciones."
            )

        codigo = service.buscar_producto(mensaje)
        if codigo:
            if not tiene_cantidad_explicita(mensaje):
                return _pedir_cantidad_producto(codigo, estado)
            return agregar_producto_al_carrito(codigo, extraer_cantidad_solicitada(mensaje), estado)

        alternativas = _alternativas_por_texto(mensaje)
        if alternativas:
            return _mensaje_alternativas(
                alternativas,
                "No tengo ese producto exactamente en el catálogo."
            )
        return "No logré identificar el producto que deseas comprar. Dime el producto y la cantidad, por ejemplo: 2 manzanas."

    for clausula in clausulas:
        condicion_texto = clausula["condicion"]
        accion_texto = clausula["accion"]

        codigo_condicion = service.buscar_producto(condicion_texto)
        if not codigo_condicion:
            respuestas.append(
                f"No pude validar la condición sobre {condicion_texto}, así que no agregué los productos dependientes de esa condición."
            )
            continue

        datos_condicion = service._catalogo.get(codigo_condicion, {})
        nombre_condicion = datos_condicion.get("nombre", condicion_texto)
        stock_condicion = int(datos_condicion.get("stock", 0))

        if stock_condicion <= 0:
            respuestas.append(
                f"Como {nombre_condicion} no está disponible en este momento, no agregué los productos condicionados a esa disponibilidad."
            )
            continue

        items_accion = extraer_items_compra(accion_texto, service)
        if not items_accion:
            respuestas.append(
                f"Sí tengo {nombre_condicion} disponible, pero no entendí qué producto querías agregar con esa condición."
            )
            continue

        resultado_accion = _procesar_items_compra(items_accion, estado, incluir_resumen=False)
        if resultado_accion and resultado_accion["interrumpido"]:
            prefijo = f"Sí tengo {nombre_condicion} disponible. "
            return ((" ".join(respuestas) + " " + prefijo + resultado_accion["response"]).strip())

        mensajes_condicionales = []
        for msg in (resultado_accion or {}).get("messages", []):
            limpio = re.sub(r"^Perfecto,\s*", "", msg).strip()
            if limpio:
                limpio = limpio[0].lower() + limpio[1:] if len(limpio) > 1 else limpio.lower()
                mensajes_condicionales.append(limpio)

        if mensajes_condicionales:
            respuestas.append(
                f"Como sí tengo {nombre_condicion} disponible, " + " ".join(mensajes_condicionales)
            )
            hubo_agregados = True

    if not respuestas:
        alternativas = _alternativas_por_texto(mensaje)
        if alternativas:
            return _mensaje_alternativas(
                alternativas,
                "No pude agregar ese producto tal cual."
            )
        return "No logré identificar un producto válido para agregar al carrito."

    cierre = f" {_resumen_carrito(estado)} ¿Deseas añadir otro producto o finalizar la compra?" if hubo_agregados else ""
    return " ".join(respuestas).strip() + cierre


def obtener_respuesta(intencion: str, mensaje: str, estado: EstadoConversacion) -> str:
    """Enrutador principal de respuestas."""

    if es_solicitud_vaciar_carrito(mensaje):
        return _vaciar_carrito(estado)

    if estado.esperando_confirmacion_contrapropuesta:
        return responder_confirmacion_contrapropuesta(mensaje, estado)

    if estado.esperando_cantidad_producto and es_consulta_exploratoria(mensaje):
        estado.esperando_cantidad_producto = False
        estado.producto_pendiente_cantidad = None

    if estado.esperando_cantidad_producto:
        return responder_cantidad_producto_pendiente(mensaje, estado)

    if estado.esperando_nombre_cliente:
        return responder_nombre_cliente_para_checkout(mensaje, estado)

    if estado.esperando_metodo_pago:
        return responder_metodo_pago_checkout(mensaje, estado)

    if estado.esperando_confirmacion_pedido:
        return confirmar_checkout(mensaje, estado)

    if estado.esperando_confirmacion_categorias and es_respuesta_afirmativa(mensaje):
        estado.esperando_confirmacion_categorias = False
        cats = service.listar_categorias()
        return "Estas son las categorías disponibles: " + ", ".join(cats) + "."

    if estado.esperando_id_pedido and extraer_id_pedido(mensaje):
        return responder_estado_pedido(mensaje, estado)

    estado.ultima_intencion = intencion

    if intencion == "Compra de productos":
        return responder_compra(mensaje, estado)
    if intencion == "Estado del pedido":
        return responder_estado_pedido(mensaje, estado)
    if intencion == "Incidencia con pedido incompleto o no recibido":
        return responder_incidencia_pedido(mensaje)
    if intencion == "Disponibilidad de productos":
        return responder_disponibilidad(mensaje, estado)
    if intencion == "Producto dañado caducado o en mal estado":
        return responder_producto_mal_estado(mensaje)
    if intencion == "Devoluciones y cambios":
        return responder_pregunta_documental(mensaje, POLITICAS["devoluciones"] + " Si quieres, puedo ayudarte a preparar la solicitud.", allowed_sources=RAG_SOURCE_HINTS["devoluciones"])
    if intencion == "Promociones y cupones":
        return responder_promociones_mvp()
    if intencion == "Métodos de pago y compra":
        return responder_pregunta_documental(mensaje, POLITICAS["pagos"], allowed_sources=RAG_SOURCE_HINTS["pagos"])
    if intencion == INTENCION_SOPORTE:
        return "Quiero evitar darte una respuesta incorrecta. Te puedo derivar con atención al cliente; antes, cuéntame brevemente qué ocurrió."

    return "No he entendido bien tu consulta. Puedo ayudarte con pedidos, compras, devoluciones, productos, promociones o soporte."


## 7. Logging del chatbot
Cada interacción se guarda en `logs_chatbot` en PostgreSQL para reentrenamiento.

In [ ]:
# ============================================================
# 7. LOGGING A POSTGRESQL
# ============================================================

def registrar_interaccion(
    mensaje: str,
    intencion: str,
    score: float,
    origen: str,
    respuesta: str,
) -> None:
    """
    Guarda la interacción en logs_chatbot.
    Si falla (sin conexión), avisa pero no interrumpe el chat.
    """
    try:
        service.guardar_log_chatbot(
            pregunta_cliente=mensaje,
            intencion_detectada=intencion,
            confianza=score,
            origen_intencion=origen,
            respuesta_bot=respuesta,
        )
    except Exception as exc:
        print(f"  ⚠ No se pudo guardar el log: {exc}")


## 8. Tests conversacionales
Ejecuta esta celda para validar el pipeline completo contra los datos reales de PostgreSQL.

In [ ]:
# ============================================================
# 8. TEST SUITE AUTOMATICO
# ============================================================

TEST_QUERIES = [
    "qué productos de la categoría lácteos tienes disponible",
    "sí",
    "hola, deseo comprar 5 manzanas y 3 leches entera",
    "agrega 2 huevos",
    "ver carrito",
    "finalizar compra",
    "Esteban Orozco, esteban@ecomarket.test, 600123456",
    "efectivo",
    "no",
    "quiero comprar 500 manzanas",
    "no",
    "quiero yogur",
    "quiero leche descremada",
    "qué promociones tienen activas?",
    "como puedo pagar y que datos necesito para cerrar el pedido?",
    "dónde está mi pedido PED-123",
    "mi pedido PED-901 llegó incompleto",
    "puedo devolver un producto fresco abierto?",
]

def ejecutar_tests():
    estado = EstadoConversacion()
    print("\n" + "=" * 65)
    print(f"  TESTS ChatBot EcoMarket v{VERSION} — fuente: PostgreSQL")
    print("=" * 65)

    for query in TEST_QUERIES:
        intencion, score, ranking, origen = procesar_mensaje(query)
        respuesta = obtener_respuesta(intencion, query, estado)

        print(f"\nUsuario : {query}")
        print(f"Intención: {intencion}  (score={score:.2f}, origen={origen})")
        print(f"Bot     : {respuesta}")
        print("-" * 65)

ejecutar_tests()


## 9. Chat interactivo

In [ ]:
# ============================================================
# 9. CHAT INTERACTIVO
# Ejecuta esta celda para iniciar la sesión conversacional.
# Escribe 'salir' para terminar.
# ============================================================

def ejecutar_chat():
    estado = EstadoConversacion()

    print("\n" + "=" * 65)
    print(f"  CHATBOT ECOMARKET v{VERSION} — PostgreSQL")
    print("=" * 65)
    print("  Escribe 'salir' para terminar.\n")

    while True:
        texto = input("Usuario: ").strip()

        if not texto:
            continue

        if _normalizar(texto) in COMANDOS_SALIDA:
            print("Bot: Hasta pronto. Cerrando sesión.")
            break

        intencion, score, _, origen = procesar_mensaje(texto)
        respuesta = obtener_respuesta(intencion, texto, estado)
        registrar_interaccion(texto, intencion, score, origen, respuesta)

        print(f"Bot: {respuesta}")
        print(f"     [intención={intencion} | score={score:.2f} | origen={origen}]")
        print("-" * 65)

ejecutar_chat()



  CHATBOT ECOMARKET v2.0.3 — PostgreSQL
  Escribe 'salir' para terminar.

Bot: Hasta pronto. Cerrando sesión.
